# 06 Final Animation Export - ADAN-86

Clean animation notebook for ADAN-86 pressure propagation.


## 1. Setup and candidate switch

In [1]:
# SETUP AND CANDIDATE SWITCH
import re
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import HTML, display

mpl.rcParams["animation.embed_limit"] = 200  # MB, for notebook preview


# Project paths — edit only if your project lives somewhere else.
PROJECT_DIR = Path(r"D:\code\adan_project")
RESULTS_DIR = PROJECT_DIR / "results"
VIDEO_OUT_DIR = RESULTS_DIR / "video_exports"


# ADAN parsed file candidates.
ADAN_PARSED_CANDIDATES = [
    PROJECT_DIR / "adan_parsed_bg.pkl",
    PROJECT_DIR / "adan_parsed.pkl",
]

# Two final animation candidates.
CANDIDATES = {
    "candidate_A_v1_8cycles": RESULTS_DIR / "optuna_E_best_trial_4_long100.pkl",
    "candidate_B_bg_v2_8cycles": RESULTS_DIR / "nb04_bg_v2_final_animation_ready_8cycles.pkl",
}

# Main switch.
#ACTIVE_CANDIDATE = "candidate_A_v1_8cycles"
ACTIVE_CANDIDATE = "candidate_B_bg_v2_8cycles"


# Video controls.
DURATION_VIDEO_S = 20.0
FPS = 30


# Waveform / network loop controls.
LOOP_MODE = "cycle_loop"
# LOOP_MODE = "recording_loop"

# Usually 1.0 s for a 60 bpm visual monitor effect.
CYCLE_PERIOD_S = 1.0

# Which cycle inside the recorded interval should be used for cycle_loop?
# 0.0 = first recorded cycle.
# For candidate B recorded from physical t=5s, this still means first LOCAL cycle of that recording.
# Try 1.0 or 2.0 if the first recorded cycle is not the cleanest.
CYCLE_START_OFFSET_S = 0.0

MONITOR_STYLE = True

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("ACTIVE_CANDIDATE:", ACTIVE_CANDIDATE)
print("LOOP_MODE:", LOOP_MODE)
print("CYCLE_PERIOD_S:", CYCLE_PERIOD_S)
print("CYCLE_START_OFFSET_S:", CYCLE_START_OFFSET_S)

print("\nCandidate files:")
for name, path in CANDIDATES.items():
    print(f"  {name:30s} | {'FOUND' if path.exists() else 'MISSING'} | {path}")

print("\nADAN parsed candidates:")
for p in ADAN_PARSED_CANDIDATES:
    print(f"  {'FOUND' if p.exists() else 'MISSING'} | {p}")


PROJECT_DIR: D:\code\adan_project
RESULTS_DIR: D:\code\adan_project\results
ACTIVE_CANDIDATE: candidate_B_bg_v2_8cycles
LOOP_MODE: cycle_loop
CYCLE_PERIOD_S: 1.0
CYCLE_START_OFFSET_S: 0.0

Candidate files:
  candidate_A_v1_8cycles         | FOUND | D:\code\adan_project\results\optuna_E_best_trial_4_long100.pkl
  candidate_B_bg_v2_8cycles      | FOUND | D:\code\adan_project\results\nb04_bg_v2_final_animation_ready_8cycles.pkl

ADAN parsed candidates:
  FOUND | D:\code\adan_project\adan_parsed_bg.pkl
  FOUND | D:\code\adan_project\adan_parsed.pkl


## 2. Load ADAN parsed topology

In [2]:
# LOAD ADAN PARSED TOPOLOGY
def load_first_existing_pickle(paths):
    """
    Load the first existing pickle file from a list of paths.

    Args:
        paths: Candidate pickle file paths.

    Returns:
        (obj, p) loaded object and path used.

    Raises:
        FileNotFoundError: If none of the paths exist.
    """
    for p in paths:
        p = Path(p)
        if p.exists():
            with open(p, "rb") as f:
                obj = pickle.load(f)
            print("Loaded:", p)
            return obj, p
    raise FileNotFoundError("None of these files exists:\n" + "\n".join(str(p) for p in paths))


def extract_order_and_topology(adan):
    """
    Extract vessel order and topology table from parsed ADAN data.

    Supports several possible key names and nested dictionary layouts,
    then normalizes the topology table to parent-child columns.

    Args:
        adan: Parsed ADAN data dictionary.

    Returns:
        (order, topology_df)
        Vessel order list and normalized topology DataFrame.

    Raises:
        TypeError: If input is not a dictionary.
        KeyError: If order or topology cannot be found.
        ValueError: If topology columns cannot be normalized.
    """
    if not isinstance(adan, dict):
        raise TypeError(f"Expected parsed ADAN pickle to be dict, got {type(adan)}")

    possible_order_keys = ["order", "vessel_order", "vessels", "vessel_names", "watch_vessel_names"]
    possible_topology_keys = ["topology_df", "topology", "edges", "edge_df"]

    order = None
    topology_df = None

    for k in possible_order_keys:
        if k in adan:
            order = list(adan[k])
            print("Found order in key:", k)
            break

    for k in possible_topology_keys:
        if k in adan:
            topology_df = adan[k]
            print("Found topology in key:", k)
            break

    # Some parsed files store topology inside nested dicts.
    if order is None or topology_df is None:
        for kk, vv in adan.items():
            if isinstance(vv, dict):
                if order is None:
                    for k in possible_order_keys:
                        if k in vv:
                            order = list(vv[k])
                            print(f"Found order in nested key: {kk}.{k}")
                            break
                if topology_df is None:
                    for k in possible_topology_keys:
                        if k in vv:
                            topology_df = vv[k]
                            print(f"Found topology in nested key: {kk}.{k}")
                            break

    if order is None:
        raise KeyError("Could not find vessel order in parsed ADAN pickle.")

    if topology_df is None:
        # Try to build from parent/child arrays if available.
        if "parent" in adan and "child" in adan:
            topology_df = pd.DataFrame({"parent": adan["parent"], "child": adan["child"]})
        else:
            raise KeyError("Could not find topology_df / topology / edges in parsed ADAN pickle.")

    if not isinstance(topology_df, pd.DataFrame):
        topology_df = pd.DataFrame(topology_df)

    # Normalize column names if needed.
    cols = list(topology_df.columns)
    if "parent" not in cols or "child" not in cols:
        lower_map = {c.lower(): c for c in cols}
        if "parent" in lower_map and "child" in lower_map:
            topology_df = topology_df.rename(columns={lower_map["parent"]: "parent", lower_map["child"]: "child"})
        elif len(cols) >= 2:
            topology_df = topology_df.rename(columns={cols[0]: "parent", cols[1]: "child"})
        else:
            raise ValueError(f"Topology dataframe columns are not usable: {cols}")

    topology_df = topology_df[["parent", "child"]].copy()

    print("Number of vessels:", len(order))
    print("Topology edges:", len(topology_df))
    display(topology_df.head())

    return order, topology_df


adan, ADAN_PARSED_PATH = load_first_existing_pickle(ADAN_PARSED_CANDIDATES)
order, topology_df = extract_order_and_topology(adan)


Loaded: D:\code\adan_project\adan_parsed_bg.pkl
Found order in key: order
Found topology in key: topology_df
Number of vessels: 103
Topology edges: 102


,parent,child
0,aortic_arch_C2,brachiocephalic_trunk_C4
1,aortic_arch_C2,aortic_arch_C46
2,brachiocephalic_trunk_C4,common_carotid_R6
3,brachiocephalic_trunk_C4,subclavian_R28
4,aortic_arch_C46,aortic_arch_C64


## 3. Robust candidate result loader

In [ ]:
# ROBUST RESULT LOADER
def find_snapshot_dict(obj, path="loaded", max_depth=6, depth=0):
    """
    Recursively find a snapshot dictionary in a loaded result object.

    Looks for dictionaries with numeric time keys and array-like
    pressure vectors.

    Args:
        obj: Object to search.
        path: Current search path label.
        max_depth: Maximum recursive search depth.
        depth: Current recursion depth.

    Returns:
        (snapshots, path) if found, otherwise (None, None).
    """
    if depth > max_depth or not isinstance(obj, dict):
        return None, None

    common_keys = [
        "snapshots", "P_snapshots", "pressure_snapshots", "snapshots_P",
        "p_snapshots", "recorded_snapshots",
    ]

    for key in common_keys:
        if key in obj and isinstance(obj[key], dict) and len(obj[key]) > 0:
            d = obj[key]
            keys = list(d.keys())[:10]
            numeric = all(isinstance(k, (int, float, np.integer, np.floating)) for k in keys)
            if numeric:
                arr = np.asarray(d[list(d.keys())[0]])
                if arr.ndim >= 1:
                    return d, f"{path}.{key}"

    # Any dict with numeric keys and array-like values.
    for k, v in obj.items():
        if isinstance(v, dict) and len(v) > 0:
            keys = list(v.keys())[:10]
            numeric = all(isinstance(x, (int, float, np.integer, np.floating)) for x in keys)
            if numeric:
                first_val = np.asarray(v[list(v.keys())[0]])
                if first_val.ndim >= 1:
                    return v, f"{path}.{k}"

    # Recursive.
    for k, v in obj.items():
        if isinstance(v, dict):
            found, found_path = find_snapshot_dict(v, f"{path}.{k}", max_depth=max_depth, depth=depth + 1)
            if found is not None:
                return found, found_path

    return None, None


def find_first_array_by_keys(obj, keys, max_depth=6, depth=0):
    """
    Recursively find the first array-like value matching preferred keys.

    Args:
        obj: Dictionary to search.
        keys: Preferred key names.
        max_depth: Maximum recursive search depth.
        depth: Current recursion depth.

    Returns:
        (array, path) if found, otherwise (None, None).
    """
    if depth > max_depth or not isinstance(obj, dict):
        return None, None

    for k in keys:
        if k in obj:
            try:
                arr = np.asarray(obj[k])
                if arr.ndim >= 1 and len(arr) > 1:
                    return arr, k
            except Exception:
                pass

    for kk, vv in obj.items():
        if isinstance(vv, dict):
            arr, path = find_first_array_by_keys(vv, keys, max_depth=max_depth, depth=depth + 1)
            if arr is not None:
                return arr, f"{kk}.{path}"

    return None, None


    """
    Load a solver result pickle and extract snapshots and time array.

    Supports direct or nested result structures and injects the found
    snapshot dictionary into the returned result for compatibility.

    Args:
        result_name: Candidate result name.
        candidates: Mapping from result names to pickle paths.

    Returns:
        (RESULT, snapshots, t_arr)
        Loaded result dictionary, snapshot dictionary, and time array.

    Raises:
        KeyError: If result name or snapshots cannot be found.
        FileNotFoundError: If the selected pickle path does not exist.
        TypeError: If the loaded object is not a dictionary.
    """
    if result_name not in candidates:
        raise KeyError(f"Unknown result_name={result_name}. Available: {list(candidates.keys())}")

    path = Path(candidates[result_name])
    if not path.exists():
        raise FileNotFoundError(path)

    with open(path, "rb") as f:
        loaded = pickle.load(f)

    print("Loaded candidate:", result_name)
    print("Path:", path)
    print("Loaded type:", type(loaded))

    if not isinstance(loaded, dict):
        raise TypeError(f"Expected dict result pickle, got {type(loaded)}")

    snapshots, snapshot_path = find_snapshot_dict(loaded)
    if snapshots is None:
        print("Top-level keys:")
        for k in loaded.keys():
            print(" -", k)
        raise KeyError("Could not find snapshot-like object in this result file.")

    # Inject for downstream compatibility.
    RESULT = loaded
    RESULT["snapshots"] = snapshots
    RESULT["candidate_name"] = result_name

    t_arr, t_path = find_first_array_by_keys(RESULT, ["t", "t_arr", "time", "times"])
    if t_arr is None:
        t_arr = np.asarray(sorted(snapshots.keys()), dtype=float)
        t_path = "snapshot keys fallback"

    snapshot_times = np.asarray(sorted([float(t) for t in snapshots.keys()]), dtype=float)
    first_t = snapshot_times[0]
    first_p = np.asarray(snapshots[first_t])

    print("Found snapshots at:", snapshot_path)
    print("Snapshots:", len(snapshots))
    print("Snapshot time range:", snapshot_times[0], "->", snapshot_times[-1])
    print("First snapshot shape:", first_p.shape)
    print("len(order):", len(order))
    if first_p.shape[0] != len(order):
        print("WARNING: first snapshot vector length != len(order)")
    print("Time array found at:", t_path, "shape:", np.asarray(t_arr).shape)

    return RESULT, snapshots, np.asarray(t_arr, dtype=float)


def compute_pressure_scale(snapshots, q_low=5, q_high=95):
    """
    Compute pressure color scale limits from snapshot values.

    Args:
        snapshots: Dictionary mapping time to pressure vectors.
        q_low: Lower percentile.
        q_high: Upper percentile.

    Returns:
        (p_low, p_high) pressure percentile limits.
    """
    ts = sorted(snapshots.keys())
    all_p = np.concatenate([np.asarray(snapshots[t]).ravel() for t in ts])
    return float(np.percentile(all_p, q_low)), float(np.percentile(all_p, q_high))


def get_brachial_signal(RESULT, snapshots, order, brachial_vessel="brachial_R34"):
    """
    Extract or reconstruct the brachial pressure waveform.

    Uses a saved brachial trace when available, otherwise builds the
    waveform from snapshot pressure vectors.

    Args:
        RESULT: Loaded solver result dictionary.
        snapshots: Snapshot dictionary.
        order: Vessel order list.
        brachial_vessel: Vessel name used for snapshot fallback.

    Returns:
        (t_wave, p_brachial) time and brachial pressure arrays.

    Raises:
        ValueError: If the fallback brachial vessel is missing.
    """
    possible_p_keys = [
        "p_brachial", "p_brachial_mid", "p_brachial_display",
        "brachial_pressure", "brachial_p", "p_brachial_R", "p_brachial_L",
    ]
    possible_t_keys = ["t", "t_arr", "time", "times"]

    p_arr, p_path = find_first_array_by_keys(RESULT, possible_p_keys)
    t_arr, t_path = find_first_array_by_keys(RESULT, possible_t_keys)

    if p_arr is not None and t_arr is not None and len(p_arr) == len(t_arr):
        print("Using saved brachial signal:", p_path)
        print("Using saved time array:", t_path)
        return np.asarray(t_arr, dtype=float), np.asarray(p_arr, dtype=float)

    # Fallback: build brachial waveform from snapshots.
    print("No aligned saved brachial signal found. Building brachial trace from snapshots.")

    if brachial_vessel not in order:
        matches = [v for v in order if "brachial" in v.lower()]
        raise ValueError(f"{brachial_vessel} not found in order. Brachial-like vessels: {matches}")

    idx = order.index(brachial_vessel)
    ts = np.asarray(sorted([float(t) for t in snapshots.keys()]), dtype=float)
    p = np.asarray([np.asarray(snapshots[t])[idx] for t in ts], dtype=float)

    print("Using vessel:", brachial_vessel)
    return ts, p


## 4. Final ADAN-86 2D layout

In [ ]:
def side_of(v):
    """
    Detect vessel side from its name.

    Args:
        v: Vessel name.

    Returns:
        +1 for right-sided vessels,
        -1 for left-sided vessels,
         0 for central or unknown vessels.
    """
    name = v.lower()
    if "_r" in name or name.endswith("_r") or "_right" in name:
        return 1
    if "_l" in name or name.endswith("_l") or "_left" in name:
        return -1
    return 0


def cnum(v):
    """
    Extract terminal C-number from a vessel name.

    Example:
        ascending_aorta_C0 -> 0

    Args:
        v: Vessel name.

    Returns:
        Integer C-number or None if missing.
    """
    m = re.search(r"_C(\d+)$", v)
    return int(m.group(1)) if m else None


In [ ]:
def build_adan86_pos2d(order):
    """
    Build 2D schematic positions for ADAN-86 vessels.

    Creates a readable graph layout for visualization and animation.
    Positions are manually assigned for known vessels, with fallback
    positions added for any missing vessels.

    Args:
        order: List of vessel names.

    Returns:
        Dictionary mapping vessel names to (x, y) plot positions.
    """
    pos2d = {}

    # Central aorta chain


    for v in order:
        name = v.lower()

        if "ascending_aorta" in name:
            pos2d[v] = (0.0, 8.2)

        elif "aortic_arch" in name:
            n = cnum(v) or 0
            y = {
                2: 7.65,
                46: 7.25,
                64: 6.85,
                94: 6.35,
            }.get(n, 6.8)
            x = {
                2: 0.0,
                46: -0.25,
                64: -0.45,
                94: 0.0,
            }.get(n, 0.0)
            pos2d[v] = (x, y)

        elif "thoracic_aorta" in name:
            n = cnum(v) or 0
            y = {
                96: 6.05,
                100: 5.60,
                104: 5.15,
                108: 4.70,
                112: 4.25,
            }.get(n, 5.0)
            pos2d[v] = (0.0, y)

        elif "abdominal_aorta" in name:
            n = cnum(v) or 0
            y = {
                114: 3.65,
                136: 3.05,
                164: 2.45,
                176: 1.85,
                188: 1.25,
                192: 0.55,
            }.get(n, 2.0)
            pos2d[v] = (0.0, y)

    # Thoracic side branches


    for v in order:
        name = v.lower()
        s = side_of(v)

        if "posterior_intercostal_t1" in name:
            pos2d[v] = (s * 1.35, 5.55)

        elif "posterior_intercostal_t2" in name:
            pos2d[v] = (s * 1.35, 4.95)


        # Manual refined positions


    manual_updates = {
        # head / neck
        "brachiocephalic_trunk_C4": (1.25, 7.45),

        "common_carotid_R6": (1.65, 8.30),
        "internal_carotid_R8": (1.45, 9.05),
        "external_carotid_T2_R26": (2.25, 8.85),
        "vertebral_R272": (2.70, 7.75),

        "common_carotid_L48": (-1.35, 8.20),
        "internal_carotid_L50": (-1.15, 9.05),
        "external_carotid_T2_L62": (-2.05, 8.80),
        "vertebral_L2": (-2.70, 7.75),


        # posterior intercostals (thoracic side branches)
        "posterior_intercostal_T1_R98": (1.35, 5.55),
        "posterior_intercostal_T1_L102": (-1.35, 5.55),
        "posterior_intercostal_T2_R106": (1.35, 4.95),
        "posterior_intercostal_T2_L110": (-1.35, 4.95),

        # right upper limb
        "subclavian_R28": (2.10, 7.00),
        "subclavian_R30": (3.00, 6.45),
        "axillary_R32": (3.75, 5.55),
        "brachial_R34": (4.10, 4.55),

        "ulnar_T2_R36": (4.72, 3.72),
        "common_interosseous_R38": (5.18, 3.18),
        "posterior_interosseous_T3_R40": (5.42, 2.50),
        "ulnar_T2_R42": (4.72, 2.95),
        "radial_T1_R44": (3.52, 3.42),


        # left upper limb
        "subclavian_L66": (-2.10, 7.00),
        "subclavian_L78": (-3.00, 6.45),
        "axillary_L80": (-3.75, 5.55),
        "brachial_L82": (-4.10, 4.55),

        "ulnar_T2_L84": (-4.72, 3.72),
        "common_interosseous_L86": (-5.18, 3.18),
        "posterior_interosseous_T3_L88": (-5.42, 2.50),
        "ulnar_T2_L90": (-4.72, 2.95),
        "radial_T1_L92": (-3.52, 3.42),

        # Celiac / splenic / gastric — upper-left compact fan

        "celiac_trunk_C116": (-0.75, 4.10),

        "left_gastric_T3_C120": (-2.05, 4.50),
        "splenic_T2_C118": (-1.75, 4.25),
        "splenic_T2_C122": (-2.45, 4.10),
        "splenic_T2_C126": (-3.00, 3.95),
        "dorsal_pancreatic_T1_C124": (-2.20, 3.70),

        # Hepatic — upper-right compact fan
        "common_hepatic_C128": (0.55, 3.30),
        "hepatic_artery_proper_C130": (1.05, 3.30),
        "hepatic_artery_proper_left_branch_C132": (1.45, 3.48),
        "hepatic_artery_proper_right_branch_C134": (1.45, 3.12),


        # LOWER SHELF: superior mesenteric / intestinal branches

        # intestinal branches — lower-left compact block
        "superior_mesenteric_T4_C138": (-1.15, 3.45),
        "superior_mesenteric_T4_C142": (-1.30, 3.20),
        "superior_mesenteric_T4_C146": (-1.30, 2.95),
        "superior_mesenteric_T4_C150": (-1.30, 2.70),
        "superior_mesenteric_T4_C154": (-1.30, 2.45),
        "superior_mesenteric_T4_C158": (-1.30, 2.20),
        "superior_mesenteric_T4_C162": (-1.30, 1.95),

        "middle_colic_T8_C140": (-2.60, 3.45),
        "jejunal_3_T10_C144": (-2.60, 3.20),
        "jejunal_6_T11_C148": (-2.60, 2.95),
        "ileocolic_T9_C152": (-2.60, 2.70),
        "ileal_4_T12_C156": (-2.60, 2.45),
        "ileal_6_T13_C160": (-2.60, 2.20),

        # Renal branches — middle shelf, clearly separated
        "renal_L166": (-1.00, 1.55),
        "renal_anterior_branch_L168": (-1.80, 1.80),
        "superior_segmental_T4_L172": (-2.35, 2.00),
        "inferior_segmental_T5_L170": (-2.35, 1.60),
        "renal_posterior_branch_T3_L174": (-1.80, 1.35),

        "renal_R178": (1.05, 2.65),
        "renal_anterior_branch_R180": (1.85, 2.85),
        "superior_segmental_T4_R182": (2.45, 3.05),
        "inferior_segmental_T5_R184": (2.45, 2.65),
        "renal_posterior_branch_T3_R186": (1.85, 2.40),


        # inferior mesenteric
        "inferior_mesenteric_T5_C190": (1.10, 1.20),

        # pelvis and legs
        "common_iliac_L194": (-0.85, 0.15),
        "internal_iliac_T1_L196": (-1.65, -0.30),
        "external_iliac_L198": (-1.05, -0.75),

        "femoral_L200": (-1.05, -1.55),
        "profundus_T2_L202": (-1.90, -1.75),
        "femoral_L204": (-1.05, -2.45),
        "popliteal_L206": (-1.05, -3.35),
        "anterior_tibial_T3_L208": (-1.75, -4.15),
        "popliteal_L210": (-1.05, -4.25),
        "tibiofibular_trunk_L212": (-1.05, -5.10),
        "posterior_tibial_T4_L214": (-1.55, -5.95),

        "common_iliac_R216": (0.85, 0.15),
        "internal_iliac_T1_R218": (1.65, -0.30),
        "external_iliac_R220": (1.05, -0.75),

        "femoral_R222": (1.05, -1.55),
        "profundus_T2_R224": (1.90, -1.75),
        "femoral_R226": (1.05, -2.45),
        "popliteal_R228": (1.05, -3.35),
        "anterior_tibial_T3_R230": (1.75, -4.15),
        "popliteal_R232": (1.05, -4.25),
        "tibiofibular_trunk_R234": (1.05, -5.10),
        "posterior_tibial_T4_R236": (1.55, -5.95),
    }

    for vessel, xy in manual_updates.items():
        if vessel in order:
            pos2d[vessel] = xy

    # Fallback for anything missing


    missing = [v for v in order if v not in pos2d]

    for k, v in enumerate(missing):
        pos2d[v] = (4.2, 2.0 - 0.25 * k)

    print("Missing positions filled by fallback:", len(missing))
    if missing:
        print(missing)

    return pos2d


In [ ]:
def edge_polyline(parent, child, pos2d):
    """
    Build a routed polyline for one vessel edge.

    Uses manual routing for difficult abdominal branches and falls
    back to a direct line for simpler edges.

    Args:
        parent: Parent vessel name.
        child: Child vessel name.
        pos2d: Dictionary of vessel 2D positions.

    Returns:
        List of (x, y) points defining the edge path.
    """
    x1, y1 = pos2d[parent]
    x2, y2 = pos2d[child]

    c = child.lower()


    # CELIAC ORIGIN
    # Route from aorta to a clean left upper shelf node
    if c == "celiac_trunk_c116":
        y_bus = 4.10
        x_bus = -0.75
        return [
            (x1, y1),
            (x1, y_bus),
            (x_bus, y_bus)
        ]

    # Upper-left celiac fan
    upper_left_celiac = [
        "left_gastric",
        "splenic",
        "dorsal_pancreatic"
    ]
    if any(k in c for k in upper_left_celiac):
        return [
            (x1, y1),
            (x1, y2),
            (x2, y2)
        ]
    if c == "common_hepatic_c128":
        x_bus = 0.55
        y_bus = 3.30
        return [
            (x1, y1),
            (x_bus, y1),
            (x_bus, y_bus),
            (x2, y_bus)
        ]
    # Hepatic side of celiac block
    hepatic_family = [
        "hepatic_artery_proper",
        "hepatic_artery_proper_left_branch",
        "hepatic_artery_proper_right_branch"
    ]
    if any(k in c for k in hepatic_family):
        return [
            (x1, y1),
            (x1, y2),
            (x2, y2)
        ]


    # 2SMA origin and intestinal comb
    if c == "superior_mesenteric_t4_c138":
        x_bus = -1.15
        return [
            (x1, y1),
            (x_bus, y1),
            (x_bus, y2),
            (x2, y2)
        ]

    if "superior_mesenteric" in c:
        return [(x1, y1), (x2, y2)]

    sma_terminals = [
        "middle_colic",
        "jejunal",
        "ileal",
        "ileocolic"
    ]
    if any(k in c for k in sma_terminals):
        return [
            (x1, y1),
            (x1, y2),
            (x2, y2)
        ]


    #  LEFT RENAL ROOT
    if c == "renal_l166":
        y_bus = 1.55
        x_bus = -1.00
        return [
            (x1, y1),
            (x1, y_bus),
            (x_bus, y_bus)
        ]

    # Left renal branches — shelves, NOT diagonal fan
    left_renal_branches = [
        "renal_anterior_branch_l168",
        "superior_segmental_t4_l172",
        "inferior_segmental_t5_l170",
        "renal_posterior_branch_t3_l174",
    ]
    if c in left_renal_branches:
        return [
            (x1, y1),
            (x1, y2),
            (x2, y2)
        ]

    # Right renal block — simple shelves
    right_renal = [
        "renal_r178",
        "renal_anterior_branch_r180",
        "superior_segmental_t4_r182",
        "inferior_segmental_t5_r184",
        "renal_posterior_branch_t3_r186",
    ]
    if c in right_renal:
        return [
            (x1, y1),
            (x1, y2),
            (x2, y2)
        ]

    # default
    return [(x1, y1), (x2, y2)]


def draw_edge(ax, parent, child, pos2d, color="darkred", lw=1.6, alpha=0.95):
    """
    Draw one routed vessel edge on an axis.

    Args:
        ax: Matplotlib axis.
        parent: Parent vessel name.
        child: Child vessel name.
        pos2d: Dictionary of vessel 2D positions.
        color: Edge color.
        lw: Line width.
        alpha: Line transparency.

    Returns:
        None.
    """
    pts = edge_polyline(parent, child, pos2d)
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]

    ax.plot(
        xs, ys,
        color=color,
        linewidth=lw,
        alpha=alpha,
        solid_capstyle="round",
        solid_joinstyle="round",
        zorder=1
    )


def build_parent_map(topology_df):
    """
    Build child-to-parent vessel mapping.

    Args:
        topology_df: DataFrame with parent and child columns.

    Returns:
        Dictionary mapping child vessel to parent vessel.
    """
    return dict(zip(topology_df["child"], topology_df["parent"]))


def draw_terminal_cap(ax, parent, child, pos2d, cap_len=0.22,
                      color="darkred", lw=1.6):
    """
    Draw a short cap at the terminal end of a vessel.

    The cap is drawn perpendicular to the final vessel segment.

    Args:
        ax: Matplotlib axis.
        parent: Parent vessel name.
        child: Terminal vessel name.
        pos2d: Dictionary of vessel 2D positions.
        cap_len: Terminal cap length.
        color: Cap color.
        lw: Line width.

    Returns:
        None.
    """
    pts = edge_polyline(parent, child, pos2d)

    # last segment direction: penultimate point -> terminal point
    (x1, y1), (x2, y2) = pts[-2], pts[-1]

    dx = x2 - x1
    dy = y2 - y1
    norm = np.hypot(dx, dy)

    if norm < 1e-12:
        return

    # unit vector along vessel
    ux = dx / norm
    uy = dy / norm

    # perpendicular unit vector
    px = -uy
    py = ux

    hx = 0.5 * cap_len * px
    hy = 0.5 * cap_len * py

    ax.plot(
        [x2 - hx, x2 + hx],
        [y2 - hy, y2 + hy],
        color=color,
        linewidth=lw,
        solid_capstyle="round",
        zorder=4
    )


def get_graph_sets(topology_df, order):
    """
    Identify terminal, root, and branch vessels from topology.

    Args:
        topology_df: DataFrame with parent and child columns.
        order: Vessel order list.

    Returns:
        (terminals, roots, branch_nodes) as sets.
    """
    parents = set(topology_df["parent"])
    children = set(topology_df["child"])

    child_count = topology_df.groupby("parent")["child"].count().to_dict()

    terminals = [v for v in order if (v in children and v not in parents)]
    roots = [v for v in order if (v in parents and v not in children)]

    # show dots for all non-terminal parent vessels
    # this is closer to the reference schematic
    branch_nodes = [v for v, n in child_count.items() if n >= 1]

    return set(terminals), set(roots), set(branch_nodes)

def plot_adan_layout(
    pos2d,
    topology_df,
    order,
    figsize=(10, 15),
    show_labels=False,
    vessel_color="darkred",
    lw=1.8,
    node_size=13,
    cap_len=0.16
):
    """
    Plot the ADAN-86 2D schematic vessel layout.

    Draws routed vessel edges, branch/root nodes, and terminal caps.

    Args:
        pos2d: Dictionary of vessel 2D positions.
        topology_df: DataFrame with parent and child columns.
        order: Vessel order list.
        figsize: Figure size.
        show_labels: Whether to show vessel labels.
        vessel_color: Vessel line and node color.
        lw: Vessel line width.
        node_size: Branch/root node marker size.
        cap_len: Terminal cap length.

    Returns:
        None.
    """
    fig, ax = plt.subplots(figsize=figsize)

    # --- draw vessel edges first
    for _, row in topology_df.iterrows():
        parent = row["parent"]
        child = row["child"]

        if parent not in pos2d or child not in pos2d:
            continue

        draw_edge(
            ax,
            parent,
            child,
            pos2d,
            color=vessel_color,
            lw=lw,
            alpha=0.98
        )

    terminals, roots, branch_nodes = get_graph_sets(topology_df, order)
    parent_map = build_parent_map(topology_df)

    # --- draw branch/root nodes only
    visible_nodes = (branch_nodes | roots) - terminals

    for vessel in order:
        if vessel not in pos2d:
            continue

        if vessel in visible_nodes:
            x, y = pos2d[vessel]
            ax.scatter(
                x,
                y,
                s=node_size,
                color=vessel_color,
                zorder=3
            )

            if show_labels:
                ax.text(
                    x + 0.04,
                    y + 0.04,
                    vessel,
                    fontsize=6,
                    alpha=0.75
                )

    # --- draw terminal caps instead of terminal dots
    for vessel in terminals:
        if vessel not in pos2d:
            continue

        parent = parent_map.get(vessel)
        if parent is None or parent not in pos2d:
            continue

        draw_terminal_cap(
            ax,
            parent=parent,
            child=vessel,
            pos2d=pos2d,
            cap_len=cap_len,
            color=vessel_color,
            lw=lw
        )

        if show_labels:
            x, y = pos2d[vessel]
            ax.text(
                x + 0.04,
                y + 0.04,
                vessel,
                fontsize=6,
                alpha=0.75
            )

    ax.set_title("ADAN-86 2D schematic layout", fontsize=14)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.show()


In [ ]:
# BUILD AND PREVIEW FINAL LAYOUT


pos2d = build_adan86_pos2d(order)

plot_adan_layout(
    pos2d=pos2d,
    topology_df=topology_df,
    order=order,
    figsize=(10, 16),
    show_labels=False,
    lw=1.25,
    node_size=7,
    cap_len=0.10,
)


## 5. Select and load active candidate

In [ ]:
# SELECT AND LOAD ACTIVE CANDIDATE
RESULT, snapshots, t_arr = load_solver_result_robust(
    ACTIVE_CANDIDATE,
    CANDIDATES,
)

t_wave, p_brachial = get_brachial_signal(
    RESULT=RESULT,
    snapshots=snapshots,
    order=order,
)

P_VMIN, P_VMAX = compute_pressure_scale(snapshots)

active_data = {
    "name": ACTIVE_CANDIDATE,
    "RESULT": RESULT,
    "snapshots": snapshots,
    "t": t_arr,
    "t_wave": t_wave,
    "p_brachial": p_brachial,
    "p_vmin": P_VMIN,
    "p_vmax": P_VMAX,
}

print("\nactive_data ready")
print("name:", active_data["name"])
print("snapshots:", len(active_data["snapshots"]))
print("snapshot range:", min(active_data["snapshots"].keys()), "->", max(active_data["snapshots"].keys()))
print("t_wave:", np.min(t_wave), "->", np.max(t_wave), "len:", len(t_wave))
print("p_brachial:", np.nanmin(p_brachial), "->", np.nanmax(p_brachial))
print("pressure scale:", P_VMIN, "->", P_VMAX)


## 6. Realtime waveform preparation

This section builds the brachial waveform directly from `snapshots` so the lower monitor panel stays aligned with the ADAN network animation.

Recommended mode: `cycle_loop`, which repeats one selected cardiac cycle across the 20-second video.


In [ ]:
# REALTIME LOOP PREPARATION — NO FLAT WAVEFORM GAPS
def get_snapshot_times(snapshots):
    """
    Return sorted snapshot times.

    Args:
        snapshots: Dictionary mapping times to snapshot arrays.

    Returns:
        Sorted snapshot times as a float NumPy array.
    """
    return np.asarray(sorted([float(t) for t in snapshots.keys()]), dtype=float)


def get_nearest_snapshot_time(snapshot_times, t):
    """
    Find the snapshot time closest to a requested time.

    Args:
        snapshot_times: Sorted snapshot time array.
        t: Requested time.

    Returns:
        Nearest snapshot time.
    """
    idx = int(np.argmin(np.abs(snapshot_times - t)))
    return float(snapshot_times[idx])


def build_brachial_trace_from_snapshots(
    snapshots,
    order,
    brachial_vessel="brachial_R34",
):
    """
    Build a brachial pressure trace from snapshot pressure vectors.

    Args:
        snapshots: Dictionary mapping times to pressure vectors.
        order: Vessel order list.
        brachial_vessel: Brachial vessel name to extract.

    Returns:
        (snapshot_times, p_brachial) arrays.

    Raises:
        ValueError: If the brachial vessel is not found.
    """
    if brachial_vessel not in order:
        matches = [v for v in order if "brachial" in v.lower()]
        raise ValueError(
            f"{brachial_vessel} not found in order. "
            f"Brachial-like vessels: {matches}"
        )

    brachial_idx = order.index(brachial_vessel)
    snapshot_times = get_snapshot_times(snapshots)

    p_brachial = np.asarray([
        np.asarray(snapshots[t])[brachial_idx]
        for t in snapshot_times
    ], dtype=float)

    return snapshot_times, p_brachial


def periodic_interp(t_source, y_source, t_query, period):
    """
    Interpolate a signal periodically over repeated cycles.

    Args:
        t_source: Source time array.
        y_source: Source signal array.
        t_query: Query time array.
        period: Loop period.

    Returns:
        Interpolated periodic signal values.

    Raises:
        ValueError: If inputs are invalid.
    """
    t_source = np.asarray(t_source, dtype=float)
    y_source = np.asarray(y_source, dtype=float)
    t_query = np.asarray(t_query, dtype=float)

    if len(t_source) != len(y_source):
        raise ValueError(f"t_source/y_source length mismatch: {len(t_source)} vs {len(y_source)}")

    # Normalize source to start at zero.
    t0 = float(t_source[0])
    ts = t_source - t0

    # Sort and remove duplicate time points.
    sort_idx = np.argsort(ts)
    ts = ts[sort_idx]
    ys = y_source[sort_idx]
    ts, unique_idx = np.unique(ts, return_index=True)
    ys = ys[unique_idx]

    if len(ts) < 2:
        raise ValueError("Need at least 2 unique source time samples for periodic interpolation.")

    period = float(period)
    if period <= 0:
        raise ValueError("period must be > 0")

    # If requested period is longer than available source, shrink to available range.
    # This avoids artificial flat tails.
    available_duration = float(ts[-1] - ts[0])
    if available_duration <= 0:
        raise ValueError("Source time duration must be > 0")

    if available_duration < 0.8 * period:
        print(
            f"WARNING: source duration {available_duration:.3f}s is much shorter than period {period:.3f}s. "
            f"Using available_duration as period."
        )
        period = available_duration

    # Keep only samples inside one period.
    keep = ts <= period
    ts = ts[keep]
    ys = ys[keep]

    # Add endpoint for smooth wrapping back to first value.
    if ts[-1] < period:
        ts_ext = np.concatenate([ts, [period]])
        ys_ext = np.concatenate([ys, [ys[0]]])
    else:
        ts_ext = ts
        ys_ext = ys
        # Force last point to wrap value if it is exactly period.
        if abs(ts_ext[-1] - period) < 1e-9:
            ys_ext[-1] = ys_ext[0]

    tq = np.mod(t_query, period)
    return np.interp(tq, ts_ext, ys_ext)


def prepare_realtime_loop_data(
    data,
    order,
    duration_video_s=20.0,
    fps=30,
    loop_mode="cycle_loop",
    cycle_period_s=1.0,
    cycle_start_offset_s=0.0,
    brachial_vessel="brachial_R34",
):
    """
    Prepare snapshot and waveform data for a looping animation.

    Builds a repeated video timeline, extracts the brachial waveform
    from snapshots, loops either one cardiac cycle or the full recording,
    and computes pressure scale limits for visualization.

    Args:
        data: Result dictionary containing snapshots.
        order: Vessel order list.
        duration_video_s: Output video duration.
        fps: Frames per second.
        loop_mode: Looping mode, either "cycle_loop" or "recording_loop".
        cycle_period_s: Cardiac cycle length for cycle looping.
        cycle_start_offset_s: Offset of selected cycle within snapshots.
        brachial_vessel: Brachial vessel name used for waveform extraction.

    Returns:
        Prepared animation data dictionary.

    Raises:
        ValueError: If snapshot timing or loop configuration is invalid.
    """
    snapshots = data["snapshots"]
    snapshot_times = get_snapshot_times(snapshots)

    physical_start = float(snapshot_times[0])
    physical_end = float(snapshot_times[-1])
    physical_duration = physical_end - physical_start

    if physical_duration <= 0:
        raise ValueError("Snapshot duration must be > 0.")

    n_video_frames = int(round(duration_video_s * fps))
    video_times = np.linspace(0.0, duration_video_s, n_video_frames)

    # Brachial trace directly from snapshots.
    wave_phys_t, wave_p = build_brachial_trace_from_snapshots(
        snapshots=snapshots,
        order=order,
        brachial_vessel=brachial_vessel,
    )

    if loop_mode == "recording_loop":
        loop_period = physical_duration
        source_t0 = physical_start
        source_t1 = physical_end

        # Network and waveform loop through the whole recording.
        physical_times_for_video = physical_start + np.mod(video_times, loop_period)

        t_source = wave_phys_t - physical_start
        p_source = wave_p

        p_video_wave = periodic_interp(
            t_source=t_source,
            y_source=p_source,
            t_query=video_times,
            period=loop_period,
        )

    elif loop_mode == "cycle_loop":
        loop_period = float(cycle_period_s)
        source_t0 = physical_start + float(cycle_start_offset_s)
        source_t1 = source_t0 + loop_period

        if source_t1 > physical_end:
            raise ValueError(
                f"Selected cycle exceeds snapshot range: "
                f"{source_t0:.3f} -> {source_t1:.3f}, snapshots end at {physical_end:.3f}. "
                "Reduce CYCLE_START_OFFSET_S or CYCLE_PERIOD_S."
            )

        # Network repeats the selected physical cycle.
        physical_times_for_video = source_t0 + np.mod(video_times, loop_period)

        # Waveform source = selected cycle only.
        mask = (wave_phys_t >= source_t0) & (wave_phys_t <= source_t1)
        if mask.sum() < 5:
            raise ValueError(
                f"Too few waveform samples in selected cycle: {source_t0:.3f} -> {source_t1:.3f}"
            )

        t_source = wave_phys_t[mask] - source_t0
        p_source = wave_p[mask]

        p_video_wave = periodic_interp(
            t_source=t_source,
            y_source=p_source,
            t_query=video_times,
            period=loop_period,
        )

    else:
        raise ValueError("loop_mode must be 'cycle_loop' or 'recording_loop'")

    # Pressure scale for network colors.
    if "p_vmin" in data and "p_vmax" in data:
        p_vmin = float(data["p_vmin"])
        p_vmax = float(data["p_vmax"])
    else:
        all_p = np.concatenate([
            np.asarray(snapshots[t]).ravel()
            for t in snapshot_times
        ])
        p_vmin = float(np.percentile(all_p, 5))
        p_vmax = float(np.percentile(all_p, 95))

    prepared = dict(data)
    prepared.update({
        "snapshot_times": snapshot_times,
        "snapshot_display_times": snapshot_times - physical_start,
        "physical_start": physical_start,
        "physical_end": physical_end,
        "physical_duration": physical_duration,

        "duration_video_s": float(duration_video_s),
        "fps": int(fps),
        "n_video_frames": int(n_video_frames),
        "video_times": video_times,
        "physical_times_for_video": physical_times_for_video,

        "loop_mode": loop_mode,
        "loop_period": float(loop_period),
        "source_t0": float(source_t0),
        "source_t1": float(source_t1),

        "t_video_wave": video_times,
        "p_video_wave": p_video_wave,

        "p_vmin": p_vmin,
        "p_vmax": p_vmax,
    })

    print("=== Realtime loop prepared ===")
    print("Candidate:", prepared.get("name", "candidate"))
    print("Snapshot physical:", physical_start, "->", physical_end)
    print("Physical duration:", physical_duration)
    print("Loop mode:", loop_mode)
    print("Loop source:", source_t0, "->", source_t1)
    print("Loop period:", loop_period)
    print("Video duration:", duration_video_s)
    print("Expected beats in video:", duration_video_s / loop_period)
    print("Wave pressure:", np.nanmin(p_video_wave), "->", np.nanmax(p_video_wave))
    print("Pressure visual scale:", p_vmin, "->", p_vmax)

    return prepared


## 7. Realtime monitor waveform drawing

The lower panel draws a continuous monitor-style brachial trace across the full video time axis. In `cycle_loop` mode this gives approximately one cardiac action per second.


In [ ]:
# REALTIME MONITOR WAVEFORM DRAWING
def setup_realtime_waveform_axis(
    ax_wave,
    prepared,
    monitor_style=True,
):
    """
    Configure the live waveform plot for animation.

    Creates waveform artists, axis styling, limits, labels,
    and the moving time marker.

    Args:
        ax_wave: Matplotlib waveform axis.
        prepared: Prepared animation data dictionary.
        monitor_style: If True, use monitor-style black background.

    Returns:
        Dictionary containing waveform plot artists.
    """
    t_video = prepared["t_video_wave"]
    p_wave = prepared["p_video_wave"]
    duration_video_s = prepared["duration_video_s"]

    if monitor_style:
        ax_wave.set_facecolor("black")
        wave_color = "lime"
        grid_color = "lime"
        marker_color = "white"
        text_color = "white"
    else:
        ax_wave.set_facecolor("white")
        wave_color = "black"
        grid_color = "gray"
        marker_color = "black"
        text_color = "black"

    wave_line, = ax_wave.plot(
        [],
        [],
        lw=2.2,
        color=wave_color,
    )

    time_marker = ax_wave.axvline(
        0.0,
        color=marker_color,
        lw=1.4,
        linestyle="--",
        alpha=0.9,
    )

    ax_wave.set_xlim(0.0, duration_video_s)
    ax_wave.set_ylim(
        float(np.nanmin(p_wave) - 5.0),
        float(np.nanmax(p_wave) + 8.0),
    )

    ax_wave.set_xlabel("Video time [s]")
    ax_wave.set_ylabel("Pressure [mmHg]")
    ax_wave.set_title(
        f"Brachial pressure waveform | {prepared['loop_mode']} | period={prepared['loop_period']:.2f}s",
        fontsize=11,
        color=text_color,
    )

    ax_wave.grid(True, color=grid_color, alpha=0.20)
    ax_wave.tick_params(colors=text_color)
    ax_wave.xaxis.label.set_color(text_color)
    ax_wave.yaxis.label.set_color(text_color)

    for spine in ax_wave.spines.values():
        spine.set_color(text_color)

    return {
        "wave_line": wave_line,
        "time_marker": time_marker,
    }


def update_realtime_waveform_axis(
    artists,
    prepared,
    current_video_t,
):
    """
    Update the waveform plot for the current animation frame.

    Draws waveform history up to the current video time and
    updates the moving time marker.

    Args:
        artists: Dictionary of waveform plot artists.
        prepared: Prepared animation data dictionary.
        current_video_t: Current animation time in seconds.

    Returns:
        List of updated Matplotlib artists.
    """
    t_video = prepared["t_video_wave"]
    p_wave = prepared["p_video_wave"]

    mask = t_video <= current_video_t

    artists["wave_line"].set_data(
        t_video[mask],
        p_wave[mask],
    )

    artists["time_marker"].set_xdata([current_video_t, current_video_t])

    return [artists["wave_line"], artists["time_marker"]]


## 8. Snapshot sanity check

Optional: plot a static pressure snapshot using raw or prepared candidate data.


In [ ]:
# SNAPSHOT PLOT FROM DATA
def plot_pressure_snapshot_from_data(
    data,
    display_t=None,
    p_vmin=None,
    p_vmax=None,
    figsize=(10, 16),
):
    """
    Plot one ADAN-86 pressure snapshot on the 2D vessel layout.

    Selects the nearest snapshot time, maps vessel pressures to colors,
    draws vessel edges, branch nodes, terminal caps, and adds a pressure
    colorbar.

    Args:
        data: Result or prepared animation dictionary containing snapshots.
        display_t: Display time to plot. If None, uses the middle snapshot.
        p_vmin: Optional lower pressure color limit.
        p_vmax: Optional upper pressure color limit.
        figsize: Matplotlib figure size.

    Returns:
        None.
    """
    snapshots = data["snapshots"]
    snapshot_times = data.get("snapshot_times", get_snapshot_times(snapshots))

    if "snapshot_display_times" in data:
        snapshot_display_times = np.asarray(data["snapshot_display_times"], dtype=float)
    else:
        snapshot_display_times = snapshot_times - float(snapshot_times[0])

    if display_t is None:
        display_t = float(snapshot_display_times[len(snapshot_display_times) // 2])

    idx_near = int(np.argmin(np.abs(snapshot_display_times - display_t)))
    physical_t = float(snapshot_times[idx_near])
    p = np.asarray(snapshots[physical_t])

    if p_vmin is None:
        p_vmin = float(data.get("p_vmin", np.percentile(p, 5)))
    if p_vmax is None:
        p_vmax = float(data.get("p_vmax", np.percentile(p, 95)))

    cmap = plt.get_cmap("plasma")
    norm = mpl.colors.Normalize(vmin=p_vmin, vmax=p_vmax)
    vessel_to_pressure = {v: p[i] for i, v in enumerate(order)}

    parents = set(topology_df["parent"])
    children = set(topology_df["child"])
    terminals = set([v for v in order if (v in children and v not in parents)])
    parent_map = dict(zip(topology_df["child"], topology_df["parent"]))

    fig, ax = plt.subplots(figsize=figsize, dpi=110)

    for _, row in topology_df.iterrows():
        parent, child = row["parent"], row["child"]
        if parent not in pos2d or child not in pos2d:
            continue
        pts = edge_polyline(parent, child, pos2d)
        xs = [pt[0] for pt in pts]
        ys = [pt[1] for pt in pts]
        ax.plot(
            xs,
            ys,
            color=cmap(norm(vessel_to_pressure[child])),
            linewidth=1.8,
            solid_capstyle="round",
            solid_joinstyle="round",
            zorder=1,
        )

    for i, vessel in enumerate(order):
        if vessel in terminals or vessel not in pos2d:
            continue
        x, y = pos2d[vessel]
        ax.scatter(x, y, s=9, color=cmap(norm(p[i])), linewidths=0, zorder=3)

    for vessel in terminals:
        parent = parent_map.get(vessel)
        if parent is None or vessel not in pos2d or parent not in pos2d:
            continue
        draw_terminal_cap(
            ax,
            parent,
            vessel,
            pos2d,
            cap_len=0.10,
            color=cmap(norm(vessel_to_pressure[vessel])),
            lw=1.8,
        )

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Pressure [mmHg]")

    ax.set_title(
        f"ADAN-86 pressure snapshot | {data.get('name', 'candidate')} | "
        f"display t={display_t:.2f}s | sim t={physical_t:.2f}s"
    )
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()


# Optional sanity check.
plot_pressure_snapshot_from_data(active_data)


## 9. Complete animation function

This is the main animation builder. It loops either a selected cycle or the whole recording and draws a continuous monitor trace across the full 20-second video.


In [ ]:
# COMPLETE ADAN-86 ANIMATION FUNCTION — V5 REALTIME LOOP
def make_adan86_animation(
    data,
    duration_video_s=20.0,
    fps=30,
    dpi=110,
    figsize=(10, 17),
    loop_mode="cycle_loop",
    cycle_period_s=1.0,
    cycle_start_offset_s=0.0,
    monitor_style=True,
    network_lw=1.8,
    node_size=9,
    cap_len=0.10,
    repeat=False,
):
    """
    Create an ADAN-86 pressure propagation animation.

    Builds a looping pressure-network animation with a synchronized
    brachial pressure waveform panel.

    Args:
        data: Result dictionary containing pressure snapshots.
        duration_video_s: Output animation duration in seconds.
        fps: Frames per second.
        dpi: Figure resolution.
        figsize: Matplotlib figure size.
        loop_mode: Looping mode, either "cycle_loop" or "recording_loop".
        cycle_period_s: Cardiac cycle duration for cycle looping.
        cycle_start_offset_s: Offset of the selected cycle in snapshots.
        monitor_style: Whether to use dark monitor-style waveform plot.
        network_lw: Network vessel line width.
        node_size: Network node marker size.
        cap_len: Terminal cap length.
        repeat: Whether animation should repeat.

    Returns:
        Dictionary containing animation object, figure, prepared data,
        and update function.
    """
    prepared = prepare_realtime_loop_data(
        data=data,
        order=order,
        duration_video_s=duration_video_s,
        fps=fps,
        loop_mode=loop_mode,
        cycle_period_s=cycle_period_s,
        cycle_start_offset_s=cycle_start_offset_s,
        brachial_vessel="brachial_R34",
    )

    snapshots = prepared["snapshots"]
    snapshot_times = prepared["snapshot_times"]
    video_times = prepared["video_times"]
    physical_times_for_video = prepared["physical_times_for_video"]
    p_vmin = prepared["p_vmin"]
    p_vmax = prepared["p_vmax"]

    # Network geometry
    vessel_to_idx = {v: i for i, v in enumerate(order)}

    segments = []
    edge_vessel_idx = []

    for _, row in topology_df.iterrows():
        parent = row["parent"]
        child = row["child"]
        if parent not in pos2d or child not in pos2d:
            continue
        segments.append(edge_polyline(parent, child, pos2d))
        edge_vessel_idx.append(vessel_to_idx[child])

    edge_vessel_idx = np.asarray(edge_vessel_idx)

    node_xy = np.asarray([pos2d[v] for v in order])
    node_x = node_xy[:, 0]
    node_y = node_xy[:, 1]

    parents = set(topology_df["parent"])
    children = set(topology_df["child"])
    terminals = set([v for v in order if (v in children and v not in parents)])
    parent_map = dict(zip(topology_df["child"], topology_df["parent"]))

    nonterminal_idx = np.asarray([i for i, v in enumerate(order) if v not in terminals])
    nonterminal_x = node_x[nonterminal_idx]
    nonterminal_y = node_y[nonterminal_idx]

    # Terminal cap geometry.
    terminal_segments = []
    terminal_vessel_idx = []

    for vessel in terminals:
        if vessel not in pos2d:
            continue
        parent = parent_map.get(vessel)
        if parent is None or parent not in pos2d:
            continue

        pts = edge_polyline(parent, vessel, pos2d)
        (x1, y1), (x2, y2) = pts[-2], pts[-1]
        dx = x2 - x1
        dy = y2 - y1
        L = np.hypot(dx, dy)
        if L < 1e-12:
            continue
        px = -dy / L
        py = dx / L
        hx = 0.5 * cap_len * px
        hy = 0.5 * cap_len * py
        terminal_segments.append([(x2 - hx, y2 - hy), (x2 + hx, y2 + hy)])
        terminal_vessel_idx.append(vessel_to_idx[vessel])

    terminal_vessel_idx = np.asarray(terminal_vessel_idx)

    # Figure
    cmap = plt.get_cmap("plasma")
    norm = mpl.colors.Normalize(vmin=p_vmin, vmax=p_vmax)

    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[5.2, 1.25], hspace=0.08)
    ax_net = fig.add_subplot(gs[0])
    ax_wave = fig.add_subplot(gs[1])

    initial_video_t = float(video_times[0])
    initial_physical_t = float(physical_times_for_video[0])
    initial_snap_t = get_nearest_snapshot_time(snapshot_times, initial_physical_t)
    initial_p = np.asarray(snapshots[initial_snap_t])

    lc = LineCollection(
        segments,
        cmap=cmap,
        norm=norm,
        linewidths=network_lw,
        capstyle="round",
        joinstyle="round",
        zorder=1,
    )
    lc.set_array(initial_p[edge_vessel_idx])
    ax_net.add_collection(lc)

    terminal_lc = LineCollection(
        terminal_segments,
        cmap=cmap,
        norm=norm,
        linewidths=network_lw,
        capstyle="round",
        zorder=4,
    )
    terminal_lc.set_array(initial_p[terminal_vessel_idx])
    ax_net.add_collection(terminal_lc)

    node_scatter = ax_net.scatter(
        nonterminal_x,
        nonterminal_y,
        c=initial_p[nonterminal_idx],
        cmap=cmap,
        norm=norm,
        s=node_size,
        linewidths=0,
        zorder=3,
    )

    title = ax_net.set_title(
        f"ADAN-86 pressure propagation | {prepared.get('name', 'candidate')} | "
        f"video t={initial_video_t:.2f}s | sim t={initial_snap_t:.2f}s",
        fontsize=13,
    )

    ax_net.set_aspect("equal")
    ax_net.axis("off")

    cbar = fig.colorbar(node_scatter, ax=ax_net, fraction=0.035, pad=0.02)
    cbar.set_label("Pressure [mmHg]")

    # Waveform panel.
    wave_artists = setup_realtime_waveform_axis(
        ax_wave=ax_wave,
        prepared=prepared,
        monitor_style=monitor_style,
    )

    update_realtime_waveform_axis(
        artists=wave_artists,
        prepared=prepared,
        current_video_t=initial_video_t,
    )

    def update(frame_idx):
        current_video_t = float(video_times[frame_idx])
        current_physical_t = float(physical_times_for_video[frame_idx])

        snap_t = get_nearest_snapshot_time(snapshot_times, current_physical_t)
        p = np.asarray(snapshots[snap_t])

        lc.set_array(p[edge_vessel_idx])
        terminal_lc.set_array(p[terminal_vessel_idx])
        node_scatter.set_array(p[nonterminal_idx])

        title.set_text(
            f"ADAN-86 pressure propagation | {prepared.get('name', 'candidate')} | "
            f"video t={current_video_t:.2f}s | sim t={snap_t:.2f}s"
        )

        wave_updated = update_realtime_waveform_axis(
            artists=wave_artists,
            prepared=prepared,
            current_video_t=current_video_t,
        )

        return [lc, terminal_lc, node_scatter, title] + wave_updated

    anim = FuncAnimation(
        fig,
        update,
        frames=prepared["n_video_frames"],
        interval=1000 / fps,
        blit=False,
        repeat=repeat,
    )

    return {
        "anim": anim,
        "fig": fig,
        "prepared": prepared,
        "update": update,
    }


## 10. Interactive preview

Run this first. It does not require FFmpeg because it renders HTML inside the notebook.


In [ ]:
# INTERACTIVE PREVIEW — ACTIVE CANDIDATE

anim_info = make_adan86_animation(
    data=active_data,
    duration_video_s=DURATION_VIDEO_S,
    fps=FPS,
    dpi=100,
    figsize=(10, 17),
    loop_mode=LOOP_MODE,
    cycle_period_s=CYCLE_PERIOD_S,
    cycle_start_offset_s=CYCLE_START_OFFSET_S,
    monitor_style=MONITOR_STYLE,
    repeat=False,
)

plt.close(anim_info["fig"])
display(HTML(anim_info["anim"].to_jshtml()))


## 11. FFmpeg check and MP4 export

MP4 export requires a real `ffmpeg.exe`. `pip install ffmpeg` is not enough.


In [ ]:
def check_ffmpeg():
    """
    Check whether a working FFmpeg executable is available.

    Searches:
    - system PATH
    - Matplotlib animation.ffmpeg_path

    Returns:
        True if FFmpeg executable is found.
        False otherwise.
    """

    ffmpeg_from_path = shutil.which("ffmpeg")
    ffmpeg_from_mpl = mpl.rcParams.get("animation.ffmpeg_path", "")

    print("ffmpeg from PATH:", ffmpeg_from_path)
    print("ffmpeg from Matplotlib:", ffmpeg_from_mpl)

    if ffmpeg_from_path is not None:
        return True

    if ffmpeg_from_mpl and Path(ffmpeg_from_mpl).exists():
        return True

    print("\nFFmpeg executable not found.")
    print("`pip install ffmpeg` installs a Python module, not ffmpeg.exe.")
    print("Install real FFmpeg, for example:")
    print("  winget install Gyan.FFmpeg")
    print("  choco install ffmpeg")
    print("or manually set mpl.rcParams['animation.ffmpeg_path'] to ffmpeg.exe")

    return False


def export_candidate_video(
    data,
    out_dir=VIDEO_OUT_DIR,
    filename_stem=None,
    duration_video_s=DURATION_VIDEO_S,
    fps=FPS,
    dpi=140,
    loop_mode=LOOP_MODE,
    cycle_period_s=CYCLE_PERIOD_S,
    cycle_start_offset_s=CYCLE_START_OFFSET_S,
    monitor_style=MONITOR_STYLE,
    bitrate=3500,
):
    """
    Export ADAN-86 pressure propagation animation to MP4.

    Workflow:
    1. Validate FFmpeg availability.
    2. Build animation object.
    3. Configure output filename.
    4. Render MP4 using Matplotlib FFMpegWriter.

    Args:
        data: Prepared solver/animation data dictionary.
        out_dir: Output directory.
        filename_stem: Optional output filename without extension.
        duration_video_s: Video duration in seconds.
        fps: Frames per second.
        dpi: Rendering DPI.
        loop_mode: "cycle_loop" or "recording_loop".
        cycle_period_s: Cardiac cycle duration used for looping.
        cycle_start_offset_s: Physical simulation offset for selected cycle.
        monitor_style: Whether to use dark monitor waveform styling.
        bitrate: MP4 encoding bitrate.

    Returns:
        Path to exported MP4 file.
    """

    if not check_ffmpeg():
        raise RuntimeError(
            "FFmpeg is not available. "
            "Install ffmpeg.exe or set "
            "mpl.rcParams['animation.ffmpeg_path']."
        )

    anim_info = make_adan86_animation(
        data=data,
        duration_video_s=duration_video_s,
        fps=fps,
        dpi=dpi,
        figsize=(10, 17),
        loop_mode=loop_mode,
        cycle_period_s=cycle_period_s,
        cycle_start_offset_s=cycle_start_offset_s,
        monitor_style=monitor_style,
        repeat=False,
    )

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    candidate_name = data.get("name", "candidate")

    if filename_stem is None:
        filename_stem = (
            f"{candidate_name}_adan86_{int(duration_video_s)}s_"
            f"{loop_mode}_period{cycle_period_s:g}_"
            f"offset{cycle_start_offset_s:g}"
        )

    out_path = out_dir / f"{filename_stem}.mp4"

    writer = FFMpegWriter(
        fps=fps,
        metadata={"title": filename_stem},
        bitrate=bitrate,
    )

    print("Saving:", out_path)

    anim_info["anim"].save(
        str(out_path),
        writer=writer,
        dpi=dpi,
    )

    plt.close(anim_info["fig"])

    print("Done:", out_path)

    return out_path


# Optional FFmpeg sanity check
check_ffmpeg()

In [ ]:
# EXPORT ACTIVE CANDIDATE MP4
out_path = export_candidate_video(
    data=active_data,
    out_dir=VIDEO_OUT_DIR,
    duration_video_s=20,
    fps=30,
    dpi=140,
    loop_mode="cycle_loop",
    cycle_period_s=1.0,
    cycle_start_offset_s=0.0,
    monitor_style=True,
)
print(out_path)
